# ⚙️ บทที่ 5 — SAM งานจริง: Large+Small และตัดถนน

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A-Phrom-A/ssb-workshop/blob/main/05_sam_production.ipynb)


> **สื่อการสอน Workshop:** การจำแนกพื้นที่ปลูกปาล์มน้ำมันด้วยภาพถ่ายดาวเทียมและ AI  
> ต่อจาก: [บทที่ 4 — SAM พื้นฐาน](04_sam_basics.ipynb)  
> ⚡ **ต้องใช้ GPU** — ไปที่เมนู Runtime → Change runtime type → T4

---

## ทฤษฎี: ทำไมต้องรัน SAM สองแบบ?

![5.1.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/5.1.png)

### กติกา Combine: Small ชนะเสมอ
![5.3.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/5.3.png)

---

## ขั้นตอนที่ 1 — Setup

In [ ]:
# เมื่อรัน cell นี้ จะมีหน้าต่างขอ permission ปรากฏขึ้น
# ให้คลิก "Connect to Google Drive" และเลือก Google Account ของคุณ
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/ssb/drive_structure/ch05_sam_production'

print("✅ เชื่อมต่อ Google Drive สำเร็จ!")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.1 — Import และ Paths
# ═══════════════════════════════════════════════════════════


import gc
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
from samgeo import SamGeo


RASTER_DIR = os.path.join(BASE_DIR, 'data')
SAT_PATH   = os.path.join(RASTER_DIR, 'for sam satelite test.tif')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
TMP_DIR    = '/content/tmp'
os.makedirs(TMP_DIR, exist_ok=True)

TMP_LARGE_TIF  = os.path.join(TMP_DIR, 'masks_large.tif')
TMP_LARGE_GPKG = os.path.join(TMP_DIR, 'masks_large.gpkg')
TMP_SMALL_TIF  = os.path.join(TMP_DIR, 'masks_small.tif')
TMP_SMALL_GPKG = os.path.join(TMP_DIR, 'masks_small.gpkg')
OUTPUT_FINAL   = '/content/output/sam_parcels_final.gpkg'
os.makedirs(os.path.dirname(OUTPUT_FINAL), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_FINAL), exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {device}')

---

## ขั้นตอนที่ 2 — กำหนด Parameters และรัน SAM สองแบบ

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.1 — กำหนด Parameters
# ═══════════════════════════════════════════════════════════

# Large: มองภาพกว้าง คลุมทุกพื้นที่
sam_kwargs_large = {
    'points_per_side': 16,
    'points_per_batch': 6,
    'pred_iou_thresh': 0.7,
    'stability_score_thresh': 0.5,
    'crop_n_layers': 0,              # ไม่ crop → มองภาพรวม
    'crop_n_points_downscale_factor': 1,
    'min_mask_region_area': 2000,    # ตัด Mask เล็กทิ้ง
}

# Small: ซอยภาพดูรายละเอียด เก็บแปลงเล็ก
sam_kwargs_small = {
    'points_per_side': 24,
    'points_per_batch': 6,
    'pred_iou_thresh': 0.8,
    'stability_score_thresh': 0.9,
    'crop_n_layers': 1,              # ซอยภาพ 3 ชั้น → รายละเอียดสูง
    'crop_n_points_downscale_factor': 2,
    'min_mask_region_area': 200,     # เก็บ Mask เล็กด้วย
}

print('✅ Parameters พร้อม')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.2 — รัน SAM Large
# ═══════════════════════════════════════════════════════════
# ⏱️ ใช้เวลา ~5-15 นาที

print('🔍 Pass 1: SAM Large...')
sam_large = SamGeo(
    model_type='vit_h', sam_kwargs=sam_kwargs_large,
    checkpoint_dir=MODEL_DIR, device=device,
)
gc.collect(); torch.cuda.empty_cache()
sam_large.generate(SAT_PATH, output=TMP_LARGE_TIF, foreground=True, unique=True)
sam_large.tiff_to_vector(TMP_LARGE_TIF, TMP_LARGE_GPKG)
gdf_large = gpd.read_file(TMP_LARGE_GPKG)
print(f'✅ Large: {len(gdf_large):,} Segment')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.3 — รัน SAM Small
# ═══════════════════════════════════════════════════════════
# ⏱️ ใช้เวลา ~10-30 นาที (มีการ crop หลายชั้น)

print('🔍 Pass 2: SAM Small...')
sam_small = SamGeo(
    model_type='vit_h', sam_kwargs=sam_kwargs_small,
    checkpoint_dir=MODEL_DIR, device=device,
)
gc.collect(); torch.cuda.empty_cache()
sam_small.generate(SAT_PATH, output=TMP_SMALL_TIF, foreground=True, unique=True)
sam_small.tiff_to_vector(TMP_SMALL_TIF, TMP_SMALL_GPKG)
gdf_small = gpd.read_file(TMP_SMALL_GPKG)
print(f'✅ Small: {len(gdf_small):,} Segment')
print(f'\n📊 Large={len(gdf_large):,} | Small={len(gdf_small):,}')

---

## ขั้นตอนที่ 3 — Combine: Small ชนะเสมอ

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.1 — ฟังก์ชัน Combine
# ═══════════════════════════════════════════════════════════

def combine_segments(gdf_small, gdf_large, min_overlap_ratio=0.02):
    A = gdf_small.copy().reset_index(drop=True)
    B = gdf_large.copy().reset_index(drop=True)
    A['geometry'] = A.geometry.buffer(0)
    B['geometry'] = B.geometry.buffer(0)
    b_geoms = list(B.geometry)
    a_geoms_out = []

    for a_geom in A.geometry:
        for j in list(B.sindex.query(a_geom, predicate='intersects')):
            b_geom = b_geoms[j]
            inter = a_geom.intersection(b_geom)
            if inter.is_empty: continue
            # ถ้า overlap < 2% → noise ขอบ ไม่ตัด
            if inter.area / a_geom.area < min_overlap_ratio: continue
            # Small ชนะ: เจาะรูออกจาก Large
            b_geoms[j] = b_geom.difference(a_geom)
        a_geoms_out.append(a_geom)

    A_out = A.copy(); A_out['geometry'] = a_geoms_out; A_out['source'] = 'small'
    B_out = B.copy(); B_out['geometry'] = b_geoms;     B_out['source'] = 'large'
    A_out = A_out[~A_out.geometry.is_empty & A_out.geometry.notna()]
    B_out = B_out[~B_out.geometry.is_empty & B_out.geometry.notna()]
    return gpd.GeoDataFrame(pd.concat([B_out, A_out], ignore_index=True), crs=A.crs)


if gdf_small.crs != gdf_large.crs:
    gdf_small = gdf_small.to_crs(gdf_large.crs)

print('🔀 กำลัง Combine...')
gdf_combined = combine_segments(gdf_small, gdf_large)
gdf_combined = gdf_combined.explode(index_parts=False).reset_index(drop=True)
gdf_combined = gdf_combined[gdf_combined.geometry.area >= 5.0]
print(f'✅ Combined: {len(gdf_combined):,} Segment')

---

## ขั้นตอนที่ 4 — ตัดถนนออก (Thinness Filter)

![5.2.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/5.2.png)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4.1 — Thinness Filter
# ═══════════════════════════════════════════════════════════

THINNESS_THRESH = 2.5  # ปรับได้ (ค่าต่ำ = เข้มงวดมากขึ้น)

def classify_linear_features(gdf, threshold):
    g = gdf.copy()
    if g.crs is None or g.crs.is_geographic:
        g = g.to_crs(g.estimate_utm_crs())
    ratio = (g.geometry.area / g.geometry.length.replace(0, float('nan'))).fillna(0)
    g['thinness_ratio'] = ratio
    g['shape_class'] = ratio.apply(lambda r: 'linear' if r < threshold else 'area')
    return g


print('🔍 กำลังตรวจจับถนน...')
gdf_cls = classify_linear_features(gdf_combined, THINNESS_THRESH)
counts = gdf_cls['shape_class'].value_counts()
print(f'  แปลงที่ดิน (area):   {counts.get("area", 0):,}')
print(f'  ถนน/คลอง (linear): {counts.get("linear", 0):,}')

gdf_area_only = gdf_cls[gdf_cls['shape_class']=='area'].drop(columns=['thinness_ratio','shape_class'])
gdf_area_only.to_file(OUTPUT_FINAL, driver='GPKG')
print(f'\n✅ บันทึกแล้ว: {len(gdf_area_only):,} แปลง')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4.2 — แสดงผลเปรียบเทียบ
# ═══════════════════════════════════════════════════════════

with rasterio.open(SAT_PATH) as src:
    bounds = src.bounds
    scale  = min(1.0, 1500 / max(src.width, src.height))
    h, w   = max(1, int(src.height * scale)), max(1, int(src.width * scale))
    rgb    = np.stack([src.read(i, out_shape=(h, w)) for i in range(1, 4)], axis=-1)
    if rgb.max() > 1: rgb = rgb / 255.0
    crs = src.crs

extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].imshow(rgb, extent=extent, origin='upper')
gdf_combined.to_crs(crs).boundary.plot(ax=axes[0], color='orange', linewidth=0.4, alpha=0.7)
axes[0].set_title(f'Combined — {len(gdf_combined):,} Segments',fontsize=11, pad=5)
axes[0].axis('off')

axes[1].imshow(rgb, extent=extent, origin='upper')
gdf_cls[gdf_cls['shape_class'] == 'linear'].to_crs(crs).boundary.plot(ax=axes[1], color='red', linewidth=0.7)
axes[1].set_title(f'Roads / Canals (Removed) — {counts.get("linear", 0):,}',fontsize=11, pad=5)
axes[1].axis('off')

axes[2].imshow(rgb, extent=extent, origin='upper')
gdf_area_only.to_crs(crs).boundary.plot(ax=axes[2], color='#35C778', linewidth=0.5)
axes[2].set_title(f'Final Result — {len(gdf_area_only):,} Plots',fontsize=11, pad=5)
axes[2].axis('off')

fig.suptitle('SAM Production Pipeline',fontsize=14,fontweight='bold',y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


---


> ➡️ **บทที่ 6:** Simplify — ทำให้เส้นขอบแปลงสวยงาม